# Proteomics CoPYlot: from intensities to differential abundance
Follow a WT versus GRK comparison in HEK293 cells (five replicates per condition). Run all cells in order for the bundled example. For your own protein-only experiment, run through **Save protein results**; the phosphosite sections are independent of the protein results.

**Start here.** From a checkout, install with `python -m pip install ".[bio]" jupyterlab`. After Beta 2 is on PyPI, use `python -m pip install "proteomicscopylot[bio]==1.0.0b2" jupyterlab`. Open this file with `python -m jupyterlab example.ipynb` from its folder, select the Python environment where you installed CoPYlot, and choose **Restart Kernel and Run All Cells**. The README explains how to copy the notebook from the wheel.

**Workflow:** settings and inputs → protein QC and statistics → phosphosite analysis → phosphoprotein aggregation. Optional demonstrations are at the end. VSN normalization and limma moderated tests require the GUI or the R functions; the Python example uses classical t-tests.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from proteomicscopylot import notebook_functions as cp

## Choose inputs and settings
Leave the settings unchanged for the example. To use your own files, set `use_example = False`, enter the paths below, and set each table's input scale to `"linear"` or `"log2"`. Relative paths start in the notebook's folder. Use wide abundance tables with one measurement column per sample. Each metadata CSV needs `sample` (an exact measurement-column name) and `condition` (group name), one row per sample. For a paired comparison also supply genuine matching `pair_id` values.

Protein plots use `ProteinNames` and GO uses `GeneNames`; the reader recognizes common aliases. For phosphosite analysis supply a collapsed site table with `UPD_seq`, `PTM_localization`, `Protein_group`, `Gene_group` and `PTM_Collapse_key`. Set `site_file = None` for protein-only data.

To reproduce this example in the GUI, use three observations **per group**, median normalization for proteins, no normalization for sites, and the non-imputed site source for aggregation. Select **filtered** data for the primary volcano comparisons at all three levels. The GUI starts with no filter or normalization.

In [ ]:
use_example = True
protein_file, protein_metadata_file = "proteins.tsv", "protein_metadata.csv"
site_file, site_metadata_file = "sites.tsv", "site_metadata.csv"
protein_scale, site_scale = "log2", "log2"

conditions = ["WT", "GRK"]  # positive log2 fold change = higher in the first group
min_present, filter_mode = 3, "per_group"
protein_normalization, site_normalization = "median", "none"
test_type = "unpaired"  # unpaired, unpaired_welch, or paired (requires pair_id)
imputation_seed = 1
filter_protein_contaminants = False
filter_phospho_contaminants = False
contaminant_categories = [1, 2, 3, 4, 5]
localization_cutoff = 0.0  # 0.75 for Class I sites
organism = "human"  # human or mouse for GO enrichment
output_dir = None  # set to Path("copylot-results") to write tables and figures

In [ ]:
def read_inputs(table_path, metadata_path, scale):
    table = cp.read_data(table_path)
    metadata = pd.read_csv(metadata_path, dtype=str)
    if not {"sample", "condition"}.issubset(metadata):
        raise ValueError("Metadata needs sample and condition columns.")
    samples = metadata["sample"]
    if metadata[["sample", "condition"]].isna().any().any() or samples.duplicated().any():
        raise ValueError("Each sample needs one unique name and a condition.")
    missing = samples[~samples.isin(table.columns)].tolist()
    if missing:
        raise ValueError(f"Metadata sample names missing from the table: {missing}")
    if len(conditions) != 2 or len(set(conditions)) != 2 or not set(conditions) <= set(metadata["condition"]):
        raise ValueError("Choose two different conditions present in the metadata.")
    if scale == "linear":
        table = cp.log2_transform_data(table, metadata)  # linear zero is missing
    elif scale == "log2":
        table[samples] = table[samples].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    else:
        raise ValueError("Input scale must be 'linear' or 'log2'.")
    return table, metadata

if use_example:
    protein_inputs = (cp.example_data_path("protein"), cp.example_data_path("protein_metadata"), "log2")
    site_inputs = (cp.example_data_path("phospho"), cp.example_data_path("phospho_metadata"), "log2")
else:
    protein_inputs = (protein_file, protein_metadata_file, protein_scale)
    site_inputs = (site_file, site_metadata_file, site_scale) if site_file is not None else None

data, meta = read_inputs(*protein_inputs)
print(f"{len(data):,} protein rows; replicates per condition:")
meta["condition"].value_counts()

# Protein

## Completeness

In [ ]:
fig = cp.missing_value_plot(data, meta, data_scale="log2")

In [ ]:
fig = cp.missing_value_heatmap(data, meta, id=False, data_scale="log2")

In [ ]:
cp.missing_sample_table(data, meta, id=False, data_scale="log2")

## Filtering and normalization

As in the app, the valid-value filter decides the rows first, then the
normalization is fitted on those rows and applied to every row, so rows with many
missing values (mostly low-abundance, missing unevenly between groups) never set
the reference. Here: at least 3 valid values in each condition, then median
centring. These are example settings; the GUI defaults to no filtering or normalization.

In [ ]:
kept = cp.filter_data(data, meta, min_present, mode=filter_mode)
kept = cp.filter_contaminants(kept, enabled=filter_protein_contaminants, categories=contaminant_categories)
if kept.empty:
    raise ValueError("No proteins remain; revise the filters.")
normalized = cp.normalize_data(data, meta, protein_normalization, fit_rows=kept.index)
filtered = normalized.loc[kept.index]
print(f"{len(filtered):,} of {len(data):,} proteins retained")

## Imputation

The default draws missing values around the 1% quantile of each sample's observed
values with 0.3 times their standard deviation; `method="numerical"` uses the Perseus
downshift instead. Keep the non-imputed matrix: the tests below count how many
imputed values each result rests on when you pass it as `observed`.

In [ ]:
imputed = cp.impute_values(filtered, meta, seed=imputation_seed)
fig = cp.impute_values(filtered, meta, ret=3, seed=imputation_seed)

## QC

QC describes the normalized, unfiltered matrix. CV and abundance are defined on
linear intensities, so they get the back-transformed values; correlation uses log2. PCA below uses the filtered, imputed matrix; select that source in the GUI to compare it.

In [ ]:
linear = cp.inverse_log2_transform_data(normalized, meta)
fig = cp.coverage_plot(normalized, meta, id=False, data_scale="log2")

In [ ]:
fig = cp.boxplot_int(normalized, meta)

In [ ]:
fig = cp.histo_int(normalized, meta)

In [ ]:
fig = cp.cov_plot(linear, meta, data_scale="linear")

In [ ]:
fig = cp.abundance_plot(linear, meta, workflow="Protein", data_scale="linear")

In [ ]:
fig = cp.corr_plot(normalized, meta, id=False, data_scale="log2")

In [ ]:
pca_figure = cp.pca_plot(imputed, meta)
cp.pca_scores(imputed, meta).head()

## Differential abundance

log2 fold changes are condition 1 minus condition 2, so positive values are higher in the first selected condition.
`test_type` chooses Student's pooled-variance test (`"unpaired"`), Welch's test
(`"unpaired_welch"`) or the paired test (`"paired"`). P-values are
Benjamini-Hochberg adjusted unless `uncorrected=True`.

In [ ]:
volcano = cp.volcano_data_f(filtered, meta, *conditions, workflow="Protein", test_type=test_type)
volcano["significance"].value_counts()

In [ ]:
fig = cp.volcano_plot(volcano, *conditions)
fig.show()

In [ ]:
# With imputed values, pass the non-imputed matrix as `observed` to count them per row.
volcano_imputed = cp.volcano_data_f(imputed, meta, *conditions, workflow="Protein", observed=filtered, test_type=test_type)
volcano_imputed[["ProteinNames", "log2FC", "adj_pval", "imputed_values"]].head()

For a paired design, supply real donor or culture identifiers in the metadata's `pair_id` column and choose `test_type = "paired"` in Settings. The HEK293 example is unpaired; row order does not establish pairing.

## ANOVA

One-way ANOVA across all conditions in the metadata; with two conditions it matches
the Student t-test.

In [ ]:
anova = cp.anova_data_f(filtered, meta)
anova.head()

In [ ]:
fig = cp.anova_plot(anova, meta)

## GO over-representation

Genes of the significant proteins against the tested proteins as background.

In [ ]:
genes = cp.different_genes(filtered, meta, *conditions, workflow="Protein", test_type=test_type)
ora = cp.enrichment_data(genes["Upregulated"] + genes["Downregulated"], organism=organism, background=genes["Background"])
ora.head()

In [ ]:
fig = cp.enrichment_analysis(genes["Upregulated"] + genes["Downregulated"], organism=organism, background=genes["Background"])

## Single protein and comparison

In [ ]:
protein_id = str(filtered["ProteinNames"].dropna().iloc[0])
fig = cp.boxplot_int_single_prot(filtered.set_index("ProteinNames", drop=False), meta, protein_id, dots=True)

In [ ]:
fig = cp.compare_prot_line(filtered, meta, conditions, [protein_id], workflow="Protein", id=False)

In [ ]:
fig = cp.correlation_xy_intensity(normalized, meta, *conditions)

In [ ]:
fig = cp.overlap_plot_condition(normalized, meta, *conditions, data_scale="log2")

## Save protein results
Set `output_dir` in Settings and rerun to save files. The default only displays results. Reusing a folder replaces files with the same names. Protein-only users can stop here.

In [ ]:
if output_dir is not None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    volcano.to_csv(output_dir / "protein-volcano.csv", index=False)
    anova.to_excel(output_dir / "protein-anova.xlsx", index=False)
    pca_figure.savefig(output_dir / "protein-pca.png", dpi=300, bbox_inches="tight")
    print(f"Saved protein outputs to {output_dir.resolve()}")

# Phosphosites
Start here after setup if you have a phosphosite table. Completeness describes the supplied sites; the comparison and the subsequent collapse use sites retained after filtering. Normalization is fitted on retained sites before applying the localization cutoff.

In [ ]:
if site_inputs is None:
    raise ValueError("Protein-only workflow is complete. Supply site_file to run the phosphosite sections.")
sites, meta_sites = read_inputs(*site_inputs)
print(f"{len(sites):,} phosphosite rows")

In [ ]:
fig = cp.simple_phos_site_plot(sites)

In [ ]:
fig = cp.phossite_coverage_plot(sites, meta_sites, id=False)

In [ ]:
fig = cp.sty_plot(sites)

In [ ]:
fig = cp.missing_value_plot(sites, meta_sites, data_scale="log2")

In [ ]:
kept_sites = cp.filter_data(sites, meta_sites, min_present, mode=filter_mode)
kept_sites = cp.filter_contaminants(kept_sites, enabled=filter_phospho_contaminants, categories=contaminant_categories)
if kept_sites.empty:
    raise ValueError("No phosphosites remain; revise the filters.")
sites_filtered = cp.normalize_data(sites, meta_sites, site_normalization, fit_rows=kept_sites.index).loc[kept_sites.index]
sites_filtered = cp.filter_localization(sites_filtered, localization_cutoff)
if sites_filtered.empty:
    raise ValueError("No sites meet the localization cutoff.")
site_volcano = cp.volcano_data_f(sites_filtered, meta_sites, *conditions, workflow="Phosphosite", test_type=test_type)
cp.volcano_plot(site_volcano, *conditions).show()

In [ ]:
site_id = str(sites_filtered["PTM_Collapse_key"].dropna().iloc[0])
fig = cp.boxplot_int_single_prot(sites_filtered.set_index("PTM_Collapse_key", drop=False), meta_sites, site_id, dots=True)

In [ ]:
protein_of_site = str(sites_filtered["Protein_group"].dropna().iloc[0]).split(";")[0].strip()
fig = cp.phossite_on_prot_regulation(sites_filtered, meta_sites, protein_of_site, conditions=conditions, id=False)

The localization cutoff in **Settings** applies to the tested and collapsed sites. At 0.75, only Class I sites (localization probability at least 0.75) are retained.

# Phosphoproteins

Sites are collapsed per protein. The default `robust_site_median` centres every site
on log2 and takes the protein median; the result keeps per-sample site-coverage
columns.

Use the filtered, observed phosphosites from the previous section, as in the GUI. Robust collapse requires values for at least half of a protein's eligible sites in a sample. Filtering sites first can therefore change both protein inclusion and abundance. Do not normalize the collapsed values again.

In [ ]:
phosprot = cp.transform_phosprot(sites_filtered, meta_sites, mode="robust_site_median",
                                 data_scale="log2", coverage_data=sites_filtered)
phosprot_filtered = cp.filter_data(phosprot, meta_sites, min_present, mode=filter_mode)
phosprot_filtered = cp.filter_contaminants(phosprot_filtered, enabled=filter_phospho_contaminants,
                                           categories=contaminant_categories)
print(f"{len(sites_filtered):,} retained sites; {len(phosprot_filtered):,} phosphoproteins after filtering")

In [ ]:
fig = cp.phossite_dist(phosprot, cutoff=0.01)

In [ ]:
phosprot_volcano = cp.volcano_data_f(phosprot_filtered, meta_sites, *conditions, workflow="Phosphoprotein", test_type=test_type)
fig = cp.volcano_plot(phosprot_volcano, *conditions)
fig.show()

In [ ]:
phosprot_id = str(phosprot_filtered["Phosphoprotein"].dropna().iloc[0])
fig = cp.compare_prot_line(phosprot_filtered, meta_sites, conditions, [phosprot_id], workflow="Phosphoprotein", id=False)

In [ ]:
if output_dir is not None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    site_volcano.to_csv(output_dir / "phosphosite-volcano.csv", index=False)
    phosprot_volcano.to_csv(output_dir / "phosphoprotein-volcano.csv", index=False)

## Optional demonstrations
Compare Welch's test with the primary protein comparison, or explore contaminant matching on a toy table. These cells do not change the analysis above.

In [ ]:
welch = cp.volcano_data_f(filtered, meta, *conditions, workflow="Protein", test_type="unpaired_welch")
welch["significance"].value_counts()

### Optional contaminant categories

| ID | Category |
|---|---|
| 1 | Laboratory proteins |
| 2 | Dust/contact proteins |
| 3 | Molecular weight markers/Standard proteins |
| 4 | Sigma-Aldrich Universal Protein Standard |
| 5 | Common viral contaminants |

The enabled example below removes albumin but retains its mixed group. Matching
uses exact protein names and, optionally, accessions; never genes. Every member
must match the selected categories. Normalization above fits the rows surviving
**both** filters, with their original indices preserved. For peptide tables, use
the same protein setting and categories on their parent-protein annotations;
keep phosphosite settings independent. A reference match is not proof of contamination.
MaxQuant `CON__` and FragPipe `contam_` prefixes are stripped only for database
matching; search-engine flags alone never remove rows. Decoy prefixes remain untouched.

The bundled metadata and attribution/licence notices are in `data/db/` in the
repository (or `proteomicscopylot/data/` in an installed wheel). See
[filtering documentation](https://github.com/proteomics-ukj/ProteomicsCoPYlot/blob/main/docs/contaminant-filtering.md).


In [ ]:
contaminant_demo = pd.DataFrame({
    "ProteinNames": ["ALBU_HUMAN", "ALBU_HUMAN;SAMPLE_PROTEIN", "SAMPLE_PROTEIN",
                     "CON__P02768", "contam_sp|P02768|ALBU_HUMAN"],
    "intensity": [100.0, 200.0, 300.0, 400.0, 500.0],
})
print(cp.filter_contaminants(contaminant_demo, enabled=True, categories=[1, 4]))
# Optional peptide use: cp.filter_contaminants(peptides, enabled=filter_protein_contaminants,
#                                         categories=contaminant_categories)